In [1]:
%pip install google-generativeai pandas openpyxl pillow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import time
import pandas as pd
from PIL import Image
import google.generativeai as genai

# SETUP & CONFIGURATION
API_KEY = "xxxxxxxxxxxxxxxxxxxx" 
# API_KEY = "MY KEY" 
genai.configure(api_key=API_KEY)

# Using gemini-2.5-flash. It is incredibly fast for image processing.
# Enforcing JSON output guarantees we don't get conversational text back.
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config={"response_mime_type": "application/json"}
)

# This prompt gives the AI the specific cultural context (Panchkula, India) and strict instructions on how to evaluate the image.
SYSTEM_PROMPT = """
You are an expert socio-economic surveyor assessing homes in Panchkula, Haryana, India. 
Look closely at the provided photo of this house and extract its structural features.

CRITICAL RULE FOR COUNTING STORIES:
Indian houses often have a small staircase head-room (mummy/barsati) on the roof. DO NOT count this as a story. 
A story must be a full functional floor spanning the majority of the house footprint.

You must return a valid JSON object with EXACTLY these six keys, IN THIS EXACT ORDER:

1. "Reasoning": (String - First, count the main functional floors. Then, look at the roof. State explicitly if you see a small staircase cabin/mummy. Finally, state the true functional story count by subtracting the cabin if necessary.)
2. "Stories": (String - e.g., "One Storied", "Two Storied", "Three Storied", "Four Storied", "Five Storied". Based strictly on your reasoning.)
3. "Roof_Type": (String - e.g., "Concrete", "Corrugated Tin/Metal", "Thatch/Tarpaulin", "Asbestos")
4. "Wall_Type": (String - e.g., "Finished Concrete/Plaster", "Exposed Brick", "Mud/Makeshift")
5. "Structural_Condition": (String - e.g., "Excellent", "Average", "Poor", "Dilapidated")
6. "Structural_Score": (Float - A strict rating between 0.0 and 1.0. A pristine concrete house is 1.0. A high-quality/well-maintained masonry house is 0.8 to 0.9. A structurally sound brick/tin house is ~0.6 to 0.7. A house with fair to moderate deterioration is 0.4 to 0.5. A collapsing makeshift hut is 0.1 to 0.3.)
"""

def analyze_image_with_gemini(image_path):
    """Sends the image and prompt to the Gemini API and parses the JSON result."""
    try:
        # Loading the image using Pillow
        img = Image.open(image_path)
        
        # Calling the Gemini API
        response = model.generate_content([SYSTEM_PROMPT, img])
        
        # Parsing the JSON response returned by Gemini
        features = json.loads(response.text)
        return features
        
    except Exception as e:
        error_msg = str(e)
        print(f"Error processing {image_path}: {error_msg}")
        # If we hit the hard quota limit, stop the script and save progress
        if "429" in error_msg or "quota" in error_msg.lower():
            raise RuntimeError("API_QUOTA_EXCEEDED")
        return None

# MAIN PIPELINE
def run_api_extraction(aggregated_excel_path, image_folder, output_excel_path):
    print(f"Loading dataset: {aggregated_excel_path}")
    df = pd.read_excel(aggregated_excel_path)
    
    vision_data = []
    processed_fids = set()

    # Resume logic: Check if we already have partial progress saved
    if os.path.exists(output_excel_path):
        existing_df = pd.read_excel(output_excel_path)
        if 'Structural_Score' in existing_df.columns:
            # Find FIDs that were successfully processed
            processed_df = existing_df.dropna(subset=['Structural_Score'])
            processed_fids = set(processed_df['FID'])
            print(f"Found {len(processed_fids)} previously processed homes. Resuming from where we left off...")
            
            # Reload existing data into our vision_data list so it isn't lost
            for _, row in processed_df.iterrows():
                vision_data.append({
                    'FID': row['FID'],
                    'Image_Found': row['Image_Found'],
                    'Stories': row.get('Stories', 'Unknown'),
                    'Roof_Type': row.get('Roof_Type', 'Unknown'),
                    'Wall_Type': row.get('Wall_Type', 'Unknown'),
                    'Structural_Condition': row.get('Structural_Condition', 'Unknown'),
                    'Structural_Score': row.get('Structural_Score', None)
                })
    
    print("\nSending images to Gemini API...")
    try:
        for index, row in df.iterrows():
            fid = row['FID']
            
            # Skipping if already processed in a previous run
            if fid in processed_fids:
                continue

            image_path = os.path.join(image_folder, f"{fid}.jpg")
            
            if os.path.exists(image_path):
                print(f"[{index+1}/{len(df)}] Analyzing FID {fid}...")
                
                features = analyze_image_with_gemini(image_path)
                
                if features:
                    vision_data.append({
                        'FID': fid,
                        'Image_Found': 'Yes',
                        'Stories': features.get('Stories', 'Unknown'),
                        'Roof_Type': features.get('Roof_Type', 'Unknown'),
                        'Wall_Type': features.get('Wall_Type', 'Unknown'),
                        'Structural_Condition': features.get('Structural_Condition', 'Unknown'),
                        'Structural_Score': features.get('Structural_Score', None)
                    })
                    
                    # IMPORTANT RATE LIMITING: 
                    # The free tier of Gemini 2.5 Flash allows ONLY 5 requests per minute.
                    # Sleeping for 15 seconds ensures we never hit the per-minute limit.
                    time.sleep(15) 
                    
                else:
                    vision_data.append({'FID': fid, 'Image_Found': 'API Error'})
            else:
                print(f"[{index+1}/{len(df)}] No image found for FID {fid}.")
                vision_data.append({'FID': fid, 'Image_Found': 'No'})

    except RuntimeError as re:
        if str(re) == "API_QUOTA_EXCEEDED":
            print("\n[STOPPED] You have hit the Google Gemini Free Tier Daily/Minute Quota.")
            print("Saving your extracted features so far so you don't lose data!")
    except Exception as e:
        print(f"\n[STOPPED] An unexpected error occurred: {e}")
        print("Saving your extracted features so far...")

    # Merging and Saving
    vision_df = pd.DataFrame(vision_data)
    print("\nMerging Gemini features into the main dataset...")
    final_df = pd.merge(df, vision_df, on='FID', how='left')
    final_df.to_excel(output_excel_path, index=False, engine='openpyxl')
    print(f"\n--- Success! Dataset saved to {output_excel_path} with {len(vision_data)} records processed ---")

if __name__ == "__main__":
    if API_KEY == "YOUR API":
        print("WAIT! You need to paste your actual Gemini API key at the top of the script first.")
    else:
        INPUT_DATASET = "Aggregated_BPL_Families.xlsx"
        IMAGE_DIRECTORY = "./IMAGES/"
        OUTPUT_DATASET = "Final_BPL_Features_Gemini.xlsx"
        
        if os.path.exists(INPUT_DATASET):
            run_api_extraction(INPUT_DATASET, IMAGE_DIRECTORY, OUTPUT_DATASET)
        else:
            print(f"Cannot find '{INPUT_DATASET}'. Please ensure the file exists.")

C:\Users\Harsh Datt\AppData\Local\Temp\ipykernel_10216\2914310601.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Loading dataset: Aggregated_BPL_Families.xlsx

Sending images to Gemini API...
[1/26] Analyzing FID 1BTJ0915...
[2/26] Analyzing FID 1FQN9637...
[3/26] Analyzing FID 1GBA1267...
[4/26] Analyzing FID 1SGR5381...
[5/26] Analyzing FID 2ECX4300...
[6/26] Analyzing FID 3CGS3602...
[7/26] Analyzing FID 3DOY3691...
[8/26] Analyzing FID 3GZT5339...
[9/26] Analyzing FID 3INZ1220...
[10/26] Analyzing FID 3VCR3405...
[11/26] Analyzing FID 4XRM1260...
[12/26] Analyzing FID 4YIX7374...
[13/26] Analyzing FID 5LTS7349...
[14/26] Analyzing FID 5NCD2023...
[15/26] Analyzing FID 5OUS9140...
[16/26] Analyzing FID 6BUT4662...
[17/26] Analyzing FID 6CFG4258...
[18/26] Analyzing FID 6SES1288...
[19/26] Analyzing FID 7KBO2244...
[20/26] Analyzing FID 7SBB1254...
[21/26] Analyzing FID 8FRN2613...
Error processing ./IMAGES/8FRN2613.jpg: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.

In [ ]:
import os
import json
import time
import pandas as pd
from PIL import Image
import google.generativeai as genai

# SETUP & CONFIGURATION
API_KEY = "xxxxxxxxxxxxxxxxxxxx" 
# API_KEY = "MY KEY" 
genai.configure(api_key=API_KEY)

# Using gemini-2.5-flash. It is incredibly fast for image processing.
# Enforcing JSON output guarantees we don't get conversational text back.
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config={"response_mime_type": "application/json"}
)

# This prompt gives the AI the specific cultural context (Panchkula, India) and strict instructions on how to evaluate the image.
SYSTEM_PROMPT = """
You are an expert socio-economic surveyor assessing homes in Panchkula, Haryana, India. 
Look closely at the provided photo of this house and extract its structural features.

CRITICAL RULE FOR COUNTING STORIES:
Indian houses often have a small staircase head-room (mummy/barsati) on the roof. DO NOT count this as a story. 
A story must be a full functional floor spanning the majority of the house footprint.

You must return a valid JSON object with EXACTLY these six keys, IN THIS EXACT ORDER:

1. "Reasoning": (String - First, count the main functional floors. Then, look at the roof. State explicitly if you see a small staircase cabin/mummy. Finally, state the true functional story count by subtracting the cabin if necessary.)
2. "Stories": (String - e.g., "One Storied", "Two Storied", "Three Storied", "Four Storied", "Five Storied". Based strictly on your reasoning.)
3. "Roof_Type": (String - e.g., "Concrete", "Corrugated Tin/Metal", "Thatch/Tarpaulin", "Asbestos")
4. "Wall_Type": (String - e.g., "Finished Concrete/Plaster", "Exposed Brick", "Mud/Makeshift")
5. "Structural_Condition": (String - e.g., "Excellent", "Average", "Poor", "Dilapidated")
6. "Structural_Score": (Float - A strict rating between 0.0 and 1.0. A pristine concrete house is 1.0. A high-quality/well-maintained masonry house is 0.8 to 0.9. A structurally sound brick/tin house is ~0.6 to 0.7. A house with fair to moderate deterioration is 0.4 to 0.5. A collapsing makeshift hut is 0.1 to 0.3.)
"""

def analyze_image_with_gemini(image_path):
    """Sends the image and prompt to the Gemini API and parses the JSON result."""
    try:
        # Loading the image using Pillow
        img = Image.open(image_path)
        
        # Calling the Gemini API
        response = model.generate_content([SYSTEM_PROMPT, img])
        
        # Parsing the JSON response returned by Gemini
        features = json.loads(response.text)
        return features
        
    except Exception as e:
        error_msg = str(e)
        print(f"Error processing {image_path}: {error_msg}")
        # If we hit the hard quota limit, stop the script and save progress
        if "429" in error_msg or "quota" in error_msg.lower():
            raise RuntimeError("API_QUOTA_EXCEEDED")
        return None

# MAIN PIPELINE
def run_api_extraction(aggregated_excel_path, image_folder, output_excel_path):
    print(f"Loading dataset: {aggregated_excel_path}")
    df = pd.read_excel(aggregated_excel_path)
    
    vision_data = []
    processed_fids = set()

    # Resume logic: Check if we already have partial progress saved
    if os.path.exists(output_excel_path):
        existing_df = pd.read_excel(output_excel_path)
        if 'Structural_Score' in existing_df.columns:
            # Find FIDs that were successfully processed
            processed_df = existing_df.dropna(subset=['Structural_Score'])
            processed_fids = set(processed_df['FID'])
            print(f"Found {len(processed_fids)} previously processed homes. Resuming from where we left off...")
            
            # Reload existing data into our vision_data list so it isn't lost
            for _, row in processed_df.iterrows():
                vision_data.append({
                    'FID': row['FID'],
                    'Image_Found': row['Image_Found'],
                    'Stories': row.get('Stories', 'Unknown'),
                    'Roof_Type': row.get('Roof_Type', 'Unknown'),
                    'Wall_Type': row.get('Wall_Type', 'Unknown'),
                    'Structural_Condition': row.get('Structural_Condition', 'Unknown'),
                    'Structural_Score': row.get('Structural_Score', None)
                })
    
    print("\nSending images to Gemini API...")
    try:
        for index, row in df.iterrows():
            fid = row['FID']
            
            # Skipping if already processed in a previous run
            if fid in processed_fids:
                continue

            image_path = os.path.join(image_folder, f"{fid}.jpg")
            
            if os.path.exists(image_path):
                print(f"[{index+1}/{len(df)}] Analyzing FID {fid}...")
                
                features = analyze_image_with_gemini(image_path)
                
                if features:
                    vision_data.append({
                        'FID': fid,
                        'Image_Found': 'Yes',
                        'Stories': features.get('Stories', 'Unknown'),
                        'Roof_Type': features.get('Roof_Type', 'Unknown'),
                        'Wall_Type': features.get('Wall_Type', 'Unknown'),
                        'Structural_Condition': features.get('Structural_Condition', 'Unknown'),
                        'Structural_Score': features.get('Structural_Score', None)
                    })
                    
                    # IMPORTANT RATE LIMITING: 
                    # The free tier of Gemini 2.5 Flash allows ONLY 5 requests per minute.
                    # Sleeping for 15 seconds ensures we never hit the per-minute limit.
                    time.sleep(15) 
                    
                else:
                    vision_data.append({'FID': fid, 'Image_Found': 'API Error'})
            else:
                print(f"[{index+1}/{len(df)}] No image found for FID {fid}.")
                vision_data.append({'FID': fid, 'Image_Found': 'No'})

    except RuntimeError as re:
        if str(re) == "API_QUOTA_EXCEEDED":
            print("\n[STOPPED] You have hit the Google Gemini Free Tier Daily/Minute Quota.")
            print("Saving your extracted features so far so you don't lose data!")
    except Exception as e:
        print(f"\n[STOPPED] An unexpected error occurred: {e}")
        print("Saving your extracted features so far...")

    # Merging and Saving
    vision_df = pd.DataFrame(vision_data)
    print("\nMerging Gemini features into the main dataset...")
    final_df = pd.merge(df, vision_df, on='FID', how='left')
    final_df.to_excel(output_excel_path, index=False, engine='openpyxl')
    print(f"\n--- Success! Dataset saved to {output_excel_path} with {len(vision_data)} records processed ---")

if __name__ == "__main__":
    if API_KEY == "YOUR API":
        print("WAIT! You need to paste your actual Gemini API key at the top of the script first.")
    else:
        INPUT_DATASET = "Aggregated_BPL_Families.xlsx"
        IMAGE_DIRECTORY = "./IMAGES/"
        OUTPUT_DATASET = "Final_BPL_Features_Gemini.xlsx"
        
        if os.path.exists(INPUT_DATASET):
            run_api_extraction(INPUT_DATASET, IMAGE_DIRECTORY, OUTPUT_DATASET)
        else:
            print(f"Cannot find '{INPUT_DATASET}'. Please ensure the file exists.")

Loading dataset: Aggregated_BPL_Families.xlsx
Found 20 previously processed homes. Resuming from where we left off...

Sending images to Gemini API...
[21/26] Analyzing FID 8FRN2613...
[22/26] Analyzing FID 8ZFK0448...
[23/26] Analyzing FID 9BXR4503...
[24/26] Analyzing FID 9JRU4960...
[25/26] Analyzing FID 9MTQ1341...
[26/26] Analyzing FID 9RAA9293...

Merging Gemini features into the main dataset...

--- Success! Dataset saved to Final_BPL_Features_Gemini.xlsx with 26 records processed ---
